<a href="https://colab.research.google.com/github/ris2002/Adult-Census-Income-MLE-NN/blob/main/Adult_Census_Income.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("uciml/adult-census-income")
file_path = os.path.join(path, "adult.csv")

print("Path to dataset files:", file_path)

Using Colab cache for faster access to the 'adult-census-income' dataset.
Path to dataset files: /kaggle/input/adult-census-income/adult.csv


EDA--Data Cleaning

In [34]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from typing import Tuple
from typing_extensions import Annotated
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

In [35]:
df=pd.read_csv(file_path)


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [37]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [38]:
df.isnull().any()


,0
age,False
workclass,False
fnlwgt,False
education,False
education.num,False
marital.status,False
occupation,False
relationship,False
race,False
sex,False


In [39]:
df=df.replace('?',np.nan)

In [40]:
df.isna().sum()


,0
age,0
workclass,1836
fnlwgt,0
education,0
education.num,0
marital.status,0
occupation,1843
relationship,0
race,0
sex,0


In [41]:
df['workclass'].mode()
df["workclass"].value_counts()


,count
workclass,
Private,22696
Self-emp-not-inc,2541
Local-gov,2093
State-gov,1298
Self-emp-inc,1116
Federal-gov,960
Without-pay,14
Never-worked,7


In [42]:
df.dropna(inplace=True)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30162 entries, 1 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             30162 non-null  int64 
 1   workclass       30162 non-null  object
 2   fnlwgt          30162 non-null  int64 
 3   education       30162 non-null  object
 4   education.num   30162 non-null  int64 
 5   marital.status  30162 non-null  object
 6   occupation      30162 non-null  object
 7   relationship    30162 non-null  object
 8   race            30162 non-null  object
 9   sex             30162 non-null  object
 10  capital.gain    30162 non-null  int64 
 11  capital.loss    30162 non-null  int64 
 12  hours.per.week  30162 non-null  int64 
 13  native.country  30162 non-null  object
 14  income          30162 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [44]:
df_columns = [
    'age','workclass', 'fnlwgt', 'education', 'education.num',
    'marital.status', 'occupation', 'relationship', 'race', 'sex',
    'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'income'
]
for col in df_columns:
  print(f'Col Name-{col}:{df[col].value_counts()}')


Col Name-age:age
36    852
31    851
33    837
34    836
37    828
     ... 
82      7
83      5
88      3
85      3
86      1
Name: count, Length: 72, dtype: int64
Col Name-workclass:workclass
Private             22286
Self-emp-not-inc     2499
Local-gov            2067
State-gov            1279
Self-emp-inc         1074
Federal-gov           943
Without-pay            14
Name: count, dtype: int64
Col Name-fnlwgt:fnlwgt
203488    13
123011    12
148995    12
121124    12
113364    12
          ..
153918     1
138145     1
312088     1
302372     1
155093     1
Name: count, Length: 20263, dtype: int64
Col Name-education:education
HS-grad         9840
Some-college    6678
Bachelors       5044
Masters         1627
Assoc-voc       1307
11th            1048
Assoc-acdm      1008
10th             820
7th-8th          557
Prof-school      542
9th              455
12th             377
Doctorate        375
5th-6th          288
1st-4th          151
Preschool         45
Name: count, dtype: int64


In [45]:

df["capital.gain"].value_counts()
df["capital.loss"].value_counts()



,count
capital.loss,
0,28735
1902,194
1977,162
1887,155
1848,50
...,...
1944,1
1411,1
1539,1


In [46]:
drop_cols=['capital.gain','capital.loss']
df.drop(columns=drop_cols,inplace=True)

In [47]:
df['income'].nunique()

2

In [48]:
numeric_cols = ['age', 'fnlwgt', 'education.num', 'hours.per.week']

categorical_cols = [
    col for col in df_columns
    if col not in numeric_cols + ['capital.gain', 'capital.loss']
]

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['age', 'fnlwgt', 'education.num', 'hours.per.week']
Categorical columns: ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country', 'income']


In [49]:
df.describe()

,age,fnlwgt,education.num,hours.per.week
count,30162.000000,3.016200e+04,30162.000000,30162.000000
mean,38.437902,1.897938e+05,10.121312,40.931238
std,13.134665,1.056530e+05,2.549995,11.979984
min,17.000000,1.376900e+04,1.000000,1.000000
25%,28.000000,1.176272e+05,9.000000,40.000000
50%,37.000000,1.784250e+05,10.000000,40.000000
75%,47.000000,2.376285e+05,13.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99.000000


In [50]:
df.head(10)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,hours.per.week,native.country,income
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,18,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,40,United-States,<=50K
7,74,State-gov,88638,Doctorate,16,Never-married,Prof-specialty,Other-relative,White,Female,20,United-States,>50K
8,68,Federal-gov,422013,HS-grad,9,Divorced,Prof-specialty,Not-in-family,White,Female,40,United-States,<=50K
10,45,Private,172274,Doctorate,16,Divorced,Prof-specialty,Unmarried,Black,Female,35,United-States,>50K
11,38,Self-emp-not-inc,164526,Prof-school,15,Never-married,Prof-specialty,Not-in-family,White,Male,45,United-States,>50K
12,52,Private,129177,Bachelors,13,Widowed,Other-service,Not-in-family,White,Female,20,United-States,>50K


EDA- Finding Relationships between cols

Pre_Processing and applying Smote

In [51]:
from imblearn.over_sampling import SMOTENC


In [52]:
X=df.drop(columns="income")
y=df['income']

In [53]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,hours.per.week,native.country,income
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,18,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,40,United-States,<=50K


In [54]:
X_train, X_test, Y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=405, stratify=y
)

In [55]:
X_train.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,hours.per.week,native.country
21027,51,Private,280093,HS-grad,9,Separated,Adm-clerical,Other-relative,White,Female,40,United-States
26432,62,Private,213321,HS-grad,9,Married-civ-spouse,Prof-specialty,Husband,White,Male,40,United-States
24087,55,Federal-gov,31965,Some-college,10,Married-civ-spouse,Adm-clerical,Husband,White,Male,40,United-States
566,41,Private,84610,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,60,United-States
1043,31,Private,106753,Assoc-voc,11,Married-civ-spouse,Craft-repair,Husband,White,Male,40,United-States


In [56]:
categorical_cols = [
        col for col in df_columns
        if col not in numeric_cols + ['capital.gain', 'capital.loss','income']
    ]

In [57]:
def label_encoder(x):
    categorical_cols = [
        col for col in df_columns
        if col not in numeric_cols + ['capital.gain', 'capital.loss','income']
    ]
    label_encoders = {}  # save encoders for later use
    for col in categorical_cols:
        le = LabelEncoder()
        x[col] = le.fit_transform(x[col])
        label_encoders[col] = le
    return x, label_encoders

In [58]:
X_train_enc, le_dict = label_encoder(X_train.copy())

In [59]:
X_train_enc

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,hours.per.week,native.country
21027,51,2,280093,11,9,5,0,2,4,0,40,38
26432,62,2,213321,11,9,2,9,0,4,1,40,38
24087,55,0,31965,15,10,2,0,0,4,1,40,38
566,41,2,84610,9,13,2,3,0,4,1,60,38
1043,31,2,106753,8,11,2,2,0,4,1,40,38
...,...,...,...,...,...,...,...,...,...,...,...,...
11573,40,2,104196,11,9,2,12,0,4,1,40,38
14148,21,2,254904,11,9,4,2,1,4,0,30,38
23310,37,2,99374,0,6,2,13,0,4,1,55,38
14392,30,5,312767,11,9,4,11,4,2,0,40,38


In [60]:
cat_indices = [X_train_enc.columns.get_loc(col) for col in categorical_cols]


In [61]:

smote_nc=SMOTENC(categorical_features=cat_indices,random_state=405)
x_train,y_train=smote_nc.fit_resample(X_train,Y_train)


In [62]:
x_train.shape

(43042, 12)

In [63]:
y_train.shape

(43042,)

In [64]:
(y_train.head)

<bound method NDFrame.head of 0        <=50K
1         >50K
2        <=50K
3         >50K
4        <=50K
         ...  
43037     >50K
43038     >50K
43039     >50K
43040     >50K
43041     >50K
Name: income, Length: 43042, dtype: object>

In [65]:
x_train.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,hours.per.week,native.country
0,51,Private,280093,HS-grad,9,Separated,Adm-clerical,Other-relative,White,Female,40,United-States
1,62,Private,213321,HS-grad,9,Married-civ-spouse,Prof-specialty,Husband,White,Male,40,United-States
2,55,Federal-gov,31965,Some-college,10,Married-civ-spouse,Adm-clerical,Husband,White,Male,40,United-States
3,41,Private,84610,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,60,United-States
4,31,Private,106753,Assoc-voc,11,Married-civ-spouse,Craft-repair,Husband,White,Male,40,United-States


In [66]:
'''

ohe_col=['workclass','education','marital.status','occupation','relationship','race','sex','native.country']

Col_Transformer=ColumnTransformer(transformers=[('OHE',OneHotEncoder(),ohe_col)])

df=Col_Transformer.fit_transform(df)
'''

"\n\nohe_col=['workclass','education','marital.status','occupation','relationship','race','sex','native.country']\n\nCol_Transformer=ColumnTransformer(transformers=[('OHE',OneHotEncoder(),ohe_col)])\n\ndf=Col_Transformer.fit_transform(df)\n"